# 03 — Job Recommender (Unsupervised: TF-IDF + Cosine Similarity)
SmartHire Phase 2 — Core Engine.

**Prerequisites:** run `python -m src.data.preprocess` then `python -m src.models.recommender` from the repo root.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import FIGURES_DIR, TOP_N_JOBS
from src.models.recommender import load_job_corpus, load_job_vectors, recommend_jobs, evaluate_recommendation
from src.data.preprocess import clean_text
from sklearn.metrics.pairwise import cosine_similarity

sns.set_style('whitegrid')
pd.set_option('display.max_colwidth', 80)

## 1. Load job corpus and pre-computed vectors

In [ ]:
job_corpus = load_job_corpus()
job_vectors, job_vectorizer = load_job_vectors()
print(f'Corpus: {len(job_corpus)} jobs  |  Vector matrix: {job_vectors.shape}')

## 2. Sample query — paste a real resume or keep the stub

In [ ]:
SAMPLE_RESUME = """
Experienced Data Scientist with 3 years in Python, machine learning, and NLP.
Proficient in pandas, NumPy, scikit-learn, TensorFlow, SQL, and data visualization.
Built classification and regression models for customer churn prediction and sentiment analysis.
"""

results = recommend_jobs(SAMPLE_RESUME, job_corpus, job_vectors, job_vectorizer, top_n=TOP_N_JOBS)
results

## 3. Match score bar chart

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
palette = sns.color_palette('Blues_d', len(results))
ax.barh(range(len(results))[::-1], results['match_score'].values, color=palette)
ax.set_yticks(range(len(results))[::-1])
ax.set_yticklabels(results['title'].values, fontsize=9)
ax.set_xlabel('Cosine Similarity Score')
ax.set_title(f'Top-{TOP_N_JOBS} Job Matches')
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'recommender_top_n_scores.png', dpi=150)
plt.show()

## 4. Full corpus score distribution

In [ ]:
resume_vec = job_vectorizer.transform([clean_text(SAMPLE_RESUME)])
all_scores = cosine_similarity(resume_vec, job_vectors).flatten()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(all_scores, bins=50, color='steelblue', edgecolor='white')
ax.axvline(results['match_score'].min(), color='red', linestyle='--',
           label=f'Top-{TOP_N_JOBS} threshold')
ax.set_xlabel('Cosine Similarity')
ax.set_ylabel('Count')
ax.set_title('Score Distribution Across Full Job Corpus')
ax.legend()
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'recommender_score_distribution.png', dpi=150)
plt.show()

## 5. Qualitative check — multiple resume types

In [ ]:
SAMPLE_RESUMES = {
    'Web Developer': 'Frontend developer with 4 years in React, JavaScript, HTML, CSS, Node.js, REST APIs.',
    'Java Backend': 'Java developer skilled in Spring Boot, microservices, Hibernate, PostgreSQL, Docker, Kubernetes.',
    'HR Professional': 'HR Manager with expertise in talent acquisition, payroll, SAP SuccessFactors, employee engagement.',
}
for label, text in SAMPLE_RESUMES.items():
    top = recommend_jobs(text, job_corpus, job_vectors, job_vectorizer, top_n=3)
    print(f'\n--- {label} ---')
    print(top[['title', 'match_score']].to_string(index=False))

## 6. Precision@K

In [ ]:
eval_rows = []
for cat, text in [('Web', SAMPLE_RESUMES['Web Developer']),
                   ('Java', SAMPLE_RESUMES['Java Backend']),
                   ('HR', SAMPLE_RESUMES['HR Professional'])]:
    row = evaluate_recommendation(text, cat, job_corpus, k=10)
    row['category'] = cat
    eval_rows.append(row)

eval_df = pd.DataFrame(eval_rows)[['category', 'k', 'hits', 'precision_at_k']]
print(eval_df.to_string(index=False))
print(f'Mean Precision@10: {eval_df["precision_at_k"].mean():.3f}')

## 7. Notes / findings (fill in after running on real data)
- Mean Precision@10: ...
- Categories with weakest recommendation quality and why: ...
- Job corpus quality issues that affected results: ...